# SIAO-CNN-OGRU: Local Training + Reliability Analysis (14-Class)


## 1) Local Setup + Runtime


In [ ]:
import os
import json
import random
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    explained_variance_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


def run_cmd(cmd: str):
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if result.returncode != 0:
        if result.stdout.strip():
            print(result.stdout)
        if result.stderr.strip():
            print(result.stderr)
        raise RuntimeError(f"Command failed: {cmd}")
    if result.stdout.strip():
        print(result.stdout.strip())
    return result


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "train_pipeline.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find repository root (missing train_pipeline.py). "
        "Open this notebook from inside siao-cnn-ogru."
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

IN_COLAB = False
print(f"REPO_ROOT={REPO_ROOT}")
print(f"Python={sys.version.split()[0]}")



In [ ]:
import platform
import shutil

print('=' * 55)
print(' ENVIRONMENT CHECK (LOCAL)')
print('=' * 55)
print(f"OS  : {platform.system()} {platform.release()}")

# GPU
try:
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
        capture_output=True, text=True, check=True
    )
    print('GPU :', result.stdout.strip())
except Exception:
    print('GPU : No NVIDIA GPU detected (or nvidia-smi unavailable)')

# RAM (cross-platform fallback)
try:
    import psutil
    vm = psutil.virtual_memory()
    total_gb = vm.total / 1024 / 1024 / 1024
    avail_gb = vm.available / 1024 / 1024 / 1024
    print(f'RAM : {total_gb:.1f} GB total, {avail_gb:.1f} GB available')
except Exception:
    print('RAM : Install psutil for detailed RAM stats (pip install psutil)')

# Disk
try:
    usage = shutil.disk_usage(str(REPO_ROOT))
    total_gb = usage.total / 1024 / 1024 / 1024
    free_gb = usage.free / 1024 / 1024 / 1024
    print(f'Disk: {free_gb:.1f} GB available of {total_gb:.1f} GB total (repo drive)')
except Exception:
    pass

print('=' * 55)



## 2) Dataset Path (Optional Zip Extract)


In [ ]:
import zipfile
from pathlib import Path

# Option A: if data is already extracted in repo
DATA_DIR = REPO_ROOT / 'data' / 'Operation_csv_data'

# Option B: set ZIP_PATH to your local zip if you want notebook to extract it
# Example: ZIP_PATH = Path('/absolute/path/to/Operation_csv_data.zip')
ZIP_PATH = None

if ZIP_PATH is not None:
    ZIP_PATH = Path(ZIP_PATH)
    if not ZIP_PATH.exists():
        raise FileNotFoundError(f"ZIP_PATH not found: {ZIP_PATH}")

    target_parent = REPO_ROOT / 'data'
    target_parent.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(target_parent)

# Auto-detect nested Operation_csv_data after extraction
candidates = list((REPO_ROOT / 'data').glob('**/Operation_csv_data'))
if candidates:
    DATA_DIR = sorted(candidates, key=lambda p: len(str(p)))[0]

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Operation_csv_data not found. Expected at: {DATA_DIR}. "
        "Put your dataset under data/Operation_csv_data or set ZIP_PATH."
    )

classes_found = sorted([p.name for p in DATA_DIR.iterdir() if p.is_dir()])
print(f"DATA_DIR={DATA_DIR}")
print(f"Accident classes found ({len(classes_found)}): {classes_found}")
DATA_DIR = str(DATA_DIR.resolve())



## 3) Data Validation & Quick Exploration


In [ ]:
from pathlib import Path
from src.siao_cnn_ogru.data.class_metadata import RESEARCH_CLASS_CODES_14
from src.siao_cnn_ogru.data.nppad_loader import NPPADDataPipeline

data_dir = Path(DATA_DIR) if "DATA_DIR" in globals() else (REPO_ROOT / "data" / "Operation_csv_data")
if not data_dir.exists():
    raise FileNotFoundError(f"Dataset directory not found: {data_dir}")

class_counts = {}
for class_dir in sorted(data_dir.iterdir()):
    if class_dir.is_dir():
        class_counts[class_dir.name] = len(list(class_dir.glob("*.csv")))

active_counts = {k: v for k, v in class_counts.items() if v > 0}
total_samples = sum(active_counts.values())

print(f"Active classes on disk: {len(active_counts)}")
print(f"Expected active classes: {len(RESEARCH_CLASS_CODES_14)}")
print(f"Total samples: {total_samples}")
print(f"Normal samples: {active_counts.get('Normal', 0)}")
print(f"TT samples: {active_counts.get('TT', 0)}")

missing = sorted(set(RESEARCH_CLASS_CODES_14) - set(active_counts))
if missing:
    raise ValueError(f"Missing expected class folders: {missing}")

dist_df = pd.DataFrame(
    [{"class": k, "samples": v} for k, v in active_counts.items()]
).sort_values("class").reset_index(drop=True)
display(dist_df)

pipeline_preview = NPPADDataPipeline(
    data_dir=str(data_dir),
    max_timesteps=100,
    normalization="none",
    active_class_codes=RESEARCH_CLASS_CODES_14,
)
X_raw, y_raw = pipeline_preview.run(use_cache=True, cache_dir=str(REPO_ROOT / "data" / "processed_raw"))

print(f"\nPreview tensor shape: {X_raw.shape}")
print(f"Unique labels in preview: {len(np.unique(y_raw))}")
print(f"Label map: {pipeline_preview.label_to_class}")

plt.figure(figsize=(12, 4))
plt.bar(dist_df["class"], dist_df["samples"], color="steelblue")
plt.title("Class Distribution (14-Class Setup)")
plt.xlabel("Class")
plt.ylabel("Samples")
plt.xticks(rotation=45, ha="right")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 4) Training Configuration


In [ ]:
FULL_TRAINING = True

if FULL_TRAINING:
    CONFIG = {
        "data_dir": str(data_dir),
        "window_size": 100,
        "stride": 25,
        "cnn_embedding_dim": 256,
        "wks_dim": 97,
        "rnn_hidden_size": 224,
        "rnn_num_layers": 2,
        "active_class_codes": RESEARCH_CLASS_CODES_14,
        "wks_pop_size": 15,
        "wks_max_iter": 30,
        "siao_pop_size": 25,
        "siao_max_iter": 80,
        "bp_epochs": 100,
        "bp_lr": 0.00157,
        "fc_dropout": 0.164,
        "weight_decay": 1.97e-05,
        "batch_size": 163,
        "use_class_weights": True,
        "label_smoothing": 0.05,
        "use_smote": True,
        "balance_to_max": False,
        "smote_target_percentile": 50,
        "smote_min_samples": 6,
        "n_folds": 10,
        "random_seed": SEED,
        "use_cache": True,
        "cache_dir": str(REPO_ROOT / "data" / "processed_raw"),
        "save_dir": str(REPO_ROOT / "results"),
    }
else:
    CONFIG = {
        "data_dir": str(data_dir),
        "window_size": 100,
        "stride": 25,
        "cnn_embedding_dim": 128,
        "wks_dim": 97,
        "rnn_hidden_size": 96,
        "rnn_num_layers": 1,
        "active_class_codes": RESEARCH_CLASS_CODES_14,
        "wks_pop_size": 6,
        "wks_max_iter": 8,
        "siao_pop_size": 6,
        "siao_max_iter": 8,
        "bp_epochs": 5,
        "bp_lr": 0.00157,
        "fc_dropout": 0.164,
        "weight_decay": 1.97e-05,
        "batch_size": 64,
        "use_class_weights": True,
        "label_smoothing": 0.05,
        "use_smote": True,
        "balance_to_max": False,
        "smote_target_percentile": 50,
        "smote_min_samples": 6,
        "n_folds": 3,
        "random_seed": SEED,
        "use_cache": True,
        "cache_dir": str(REPO_ROOT / "data" / "processed_raw"),
        "save_dir": str(REPO_ROOT / "results"),
    }

for k, v in CONFIG.items():
    print(f"{k}: {v}")



## 5) Train Model


In [ ]:
from train_pipeline import run_complete_pipeline

results = run_complete_pipeline(**CONFIG)

print(f"\n{'='*60}")
print(f"Average Accuracy: {results['avg_accuracy']*100:.2f}%")
print(f"Std Accuracy: {results['std_accuracy']*100:.2f}%")
print(f"Average Macro-F1: {results['avg_macro_f1']*100:.2f}%")
print(f"Class codes: {results.get('class_codes', [])}")
print(f"{'='*60}")


In [ ]:
results_dir = REPO_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)

training_summary = {
    "avg_accuracy": float(results["avg_accuracy"]),
    "std_accuracy": float(results["std_accuracy"]),
    "avg_macro_f1": float(results["avg_macro_f1"]),
    "fold_accuracies": [float(x) for x in results["fold_accuracies"]],
    "fold_macro_f1": [float(x) for x in results["fold_macro_f1"]],
    "class_codes": results.get("class_codes", []),
    "class_metadata": results.get("class_metadata", []),
    "model_design": "SIAO-CNN-OGRU",
}

summary_path = results_dir / "training_summary_ogru_14class.json"
summary_path.write_text(json.dumps(training_summary, indent=2))
print(f"Saved: {summary_path}")

fold_ids = np.arange(1, len(training_summary["fold_accuracies"]) + 1)
acc = np.array(training_summary["fold_accuracies"]) * 100.0
f1 = np.array(training_summary["fold_macro_f1"]) * 100.0

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(fold_ids, acc, marker="o")
plt.title("Fold Accuracy (%)")
plt.xlabel("Fold")
plt.ylabel("Accuracy")
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(fold_ids, f1, marker="o", color="tab:orange")
plt.title("Fold Macro-F1 (%)")
plt.xlabel("Fold")
plt.ylabel("Macro-F1")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
import glob
from PIL import Image

plot_files = sorted(glob.glob(str(results_dir / "plots" / "*.png")))
if plot_files:
    for pf in plot_files:
        img = Image.open(pf)
        plt.figure(figsize=(10, 8))
        plt.imshow(img)
        plt.title(Path(pf).name)
        plt.axis("off")
        plt.tight_layout()
        plt.show()
else:
    print("No plot images found under results/plots")


## 6) Reliability Analysis


In [ ]:
y_true = np.asarray(results.get("oof_y_true", []), dtype=np.int64)
y_pred = np.asarray(results.get("oof_y_pred", []), dtype=np.int64)
class_codes = results.get("class_codes", [])

if y_true.size == 0 or y_pred.size == 0:
    raise RuntimeError("OOF predictions are missing. Please rerun training.")

if "Normal" in class_codes:
    normal_id = class_codes.index("Normal")
    true_failure_events = (y_true != normal_id).astype(np.int64)
    pred_failure_events = (y_pred != normal_id).astype(np.int64)
    failure_mode = f"Class-based failure stream with Normal label id={normal_id}"
else:
    true_failure_events = (y_true != y_pred).astype(np.int64)
    pred_failure_events = true_failure_events.copy()
    failure_mode = "Prediction-error proxy stream (Normal class not present)"

print(f"Failure mode: {failure_mode}")
print(f"Failure events count (target): {true_failure_events.sum()} / {len(true_failure_events)}")

time_step_hours = 1.0


def dynamic_reliability(failure_events: np.ndarray, dt_hours: float = 1.0):
    failure_events = np.asarray(failure_events, dtype=np.float64)
    t = np.arange(1, len(failure_events) + 1, dtype=np.float64) * dt_hours
    cumulative_failures = np.cumsum(failure_events)
    lambda_t = cumulative_failures / t
    lambda_t = np.clip(lambda_t, 1e-12, None)
    mttf_t = 1.0 / lambda_t
    reliability_t = np.exp(-t / mttf_t)
    return t, lambda_t, mttf_t, reliability_t


t, lambda_true, mttf_true, reliability_true = dynamic_reliability(true_failure_events, time_step_hours)
_, lambda_ours, mttf_ours, reliability_ours = dynamic_reliability(pred_failure_events, time_step_hours)

reliability_const = np.full_like(reliability_true, reliability_true.mean())
coef = np.polyfit(t, reliability_true, deg=1)
reliability_linear = np.clip(np.polyval(coef, t), 0.0, 1.0)

window = min(24, len(pred_failure_events))
kernel = np.ones(window) / window
ma_failure_prob = np.convolve(pred_failure_events, kernel, mode="same")
lambda_ma = np.clip(ma_failure_prob / time_step_hours, 1e-12, None)
mttf_ma = 1.0 / lambda_ma
reliability_ma = np.exp(-t / mttf_ma)


def score_curve(y_ref: np.ndarray, y_hat: np.ndarray):
    return {
        "RMSE": float(np.sqrt(mean_squared_error(y_ref, y_hat))),
        "MAE": float(mean_absolute_error(y_ref, y_hat)),
        "EVS": float(explained_variance_score(y_ref, y_hat)),
        "R2": float(r2_score(y_ref, y_hat)),
    }


scores = {
    "SIAO-CNN-OGRU Reliability": score_curve(reliability_true, reliability_ours),
    "Baseline Constant": score_curve(reliability_true, reliability_const),
    "Baseline Linear": score_curve(reliability_true, reliability_linear),
    "Baseline MovingAverage": score_curve(reliability_true, reliability_ma),
}

score_df = pd.DataFrame(scores).T.sort_values(["RMSE", "MAE"], ascending=True)
display(score_df)


In [ ]:
plt.figure(figsize=(14, 10))

plt.subplot(2, 2, 1)
plt.plot(t, lambda_true, label="Target λ(t)", linewidth=2)
plt.plot(t, lambda_ours, label="Predicted λ(t)", linewidth=2, alpha=0.8)
plt.title("Dynamic Failure Rate λ(t)")
plt.xlabel("Operating Time (hours)")
plt.ylabel("Failure Rate")
plt.grid(alpha=0.3)
plt.legend()

plt.subplot(2, 2, 2)
plt.plot(t, mttf_true, label="Target MTTF(t)", linewidth=2)
plt.plot(t, mttf_ours, label="Predicted MTTF(t)", linewidth=2, alpha=0.8)
plt.title("Dynamic Mean Time To Failure")
plt.xlabel("Operating Time (hours)")
plt.ylabel("MTTF (hours)")
plt.grid(alpha=0.3)
plt.legend()

plt.subplot(2, 2, 3)
plt.plot(t, reliability_true, label="Target Reliability", linewidth=2)
plt.plot(t, reliability_ours, label="SIAO-CNN-OGRU", linewidth=2)
plt.plot(t, reliability_const, label="Constant Baseline", linestyle="--")
plt.plot(t, reliability_linear, label="Linear Baseline", linestyle="--")
plt.plot(t, reliability_ma, label="MA Baseline", linestyle="--")
plt.title("Dynamic Reliability Curves")
plt.xlabel("Operating Time (hours)")
plt.ylabel("Reliability R(t)")
plt.ylim(0, 1.02)
plt.grid(alpha=0.3)
plt.legend()

plt.subplot(2, 2, 4)
rmse_values = score_df["RMSE"].values
labels = score_df.index.tolist()
plt.barh(labels, rmse_values)
plt.title("RMSE Comparison (lower is better)")
plt.xlabel("RMSE")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
reliability_summary = {
    "failure_mode": failure_mode,
    "time_step_hours": time_step_hours,
    "scores": {k: {m: float(v) for m, v in d.items()} for k, d in scores.items()},
    "final_failure_rate": float(lambda_ours[-1]),
    "final_mttf": float(mttf_ours[-1]),
    "final_reliability": float(reliability_ours[-1]),
}

reliability_path = results_dir / "reliability_summary_ogru_14class.json"
reliability_path.write_text(json.dumps(reliability_summary, indent=2))
print(f"Saved: {reliability_path}")
reliability_summary


## 7) Package Results


In [ ]:
from pathlib import Path
import shutil

results_dir = REPO_ROOT / 'results'
if not results_dir.exists():
    raise FileNotFoundError(f"Results directory not found: {results_dir}")

archive_base = REPO_ROOT / 'results_ogru_14class_local'
zip_path = Path(shutil.make_archive(str(archive_base), 'zip', root_dir=results_dir))

print(f"Packaged results zip: {zip_path}")
print("You can now share/upload this zip file.")

